# ROGII Last Value Baseline

Predict the unknown zone with the last known `TVT_input` value and write `submission.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

sample_paths = sorted(Path('/kaggle/input').rglob('sample_submission.csv'))
if not sample_paths:
    raise FileNotFoundError('sample_submission.csv was not mounted under /kaggle/input')
data_root = sample_paths[0].parent
sample = pd.read_csv(sample_paths[0])
predictions = {}
horizontal_files = sorted((data_root / 'test').glob('*__horizontal_well.csv'))
for path in horizontal_files:
    df = pd.read_csv(path).reset_index(drop=True)
    values = pd.to_numeric(df['TVT_input'], errors='coerce').to_numpy(dtype=float)
    known = values[np.isfinite(values)]
    if len(known) == 0:
        raise ValueError(f'No known TVT_input values: {path}')
    last_value = float(known[-1])
    well = path.stem.split('__horizontal_well', 1)[0]
    for row_index in np.flatnonzero(~np.isfinite(values)):
        predictions[f'{well}_{int(row_index)}'] = last_value

submission = sample[['id']].copy()
submission['tvt'] = submission['id'].map(predictions)
if submission['tvt'].isna().any():
    raise ValueError(f'Missing predictions for {int(submission.tvt.isna().sum())} rows')
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'wells={len(horizontal_files)} rows={len(submission)}')
print(submission.head())